# Temporal v3 Dummy Batch Shape Check

Run this on the workstation to verify the v3 model input/output contract without loading the ticker-month cache.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = next(parent for parent in Path.cwd().resolve().parents if (parent / 'research').exists()) if Path.cwd().name != 'quant-research-workbench' else Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import torch
from research.temporal_event_model.v3.config import ModelConfig
from research.temporal_event_model.v3.data import make_dummy_temporal_batch
from research.temporal_event_model.v3.losses import compute_loss
from research.temporal_event_model.v3.model import TemporalEventModelV3

config = ModelConfig(d_model=64, event_stream_length=128, event_layers=1, event_heads=4, fusion_layers=1, fusion_heads=4, xbrl_max_items=64, corporate_action_max_items=16)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch = make_dummy_temporal_batch(model_config=config, batch_size=4, device=device)
model = TemporalEventModelV3(config).to(device)
output = model(batch.x)
loss = compute_loss(output, batch)
print('device', device)
print('loss', float(loss.loss.detach().cpu()))

In [ ]:
def show_tree(name, value, indent=0):
    pad = '  ' * indent
    if torch.is_tensor(value):
        print(f'{pad}{name}: tensor shape={tuple(value.shape)} dtype={value.dtype}')
    elif isinstance(value, dict):
        print(f'{pad}{name}: dict')
        for key in sorted(value):
            show_tree(str(key), value[key], indent + 1)
    else:
        print(f'{pad}{name}: {type(value).__name__}')

show_tree('x', batch.x)
show_tree('y', batch.y)
show_tree('future_bar_values', output.future_bar_values)
show_tree('intraday_logits', output.intraday_logits)
show_tree('corporate_action_logits', output.corporate_action_logits)
print(loss.metrics)